Test notebook for basic environment functions.

env.reset() launches a research-friendly version of the game. To start manually instead, run:

AIR_SDK_HOME="$HOME/Developer/AIRSDK_51.3.3" bash tools/macos/run_macos.sh

in the terminal

In [1]:
# Player declarations are controller declarations:
from ssf2_rl.policy.bots import Agent, ZeroBot, FollowBot, ScriptedBot
from ssf2_rl.game.catalog import Character, Stage
from ssf2_rl.game.players import CPU, Human
from ssf2_rl import NOOP, LEFT, RIGHT, DOWN, SPECIAL, ATTACK
from ssf2_rl.env.gym_env import SSF2Env

try:
    env.close()
except NameError:
    pass
env = SSF2Env(step_timeout=5.0)  # minimal JSON is the compatibility default

In [2]:
# Programmatic match setup: declare every slot and the stage in reset().
obs, info = env.reset(
    players={
        1: Agent(Character.Marth),
        2: ZeroBot(Character.ZeroSuitSamus)},
    stage=Stage.bf,
)
print(env.describe_matchup())
print("\nframe:", info["frame"], "| me:", info["me"]["name"], "| opp:", info["opp"]["name"])

# ZeroBot's latest zero mask is held until Python replaces it; it cannot fall
# through to the backing native level-9 AI while Python is idle.
full = env.request_full_state()
zss = next(char for char in full["chars"] if char["id"] == 2)
assert zss["controls"] == 0
print("ZeroBot held controls:", zss["controls"])

[ssf2_rl] Launching SSF2 via /Users/cachemiss/Developer/AIRSDK_51.3.3/bin/adl (log: /Users/cachemiss/Documents/projects/reflash2-fork/reflash2/.macos/adl.log) ...
stage: battlefield
P1: external agent (step-driven), character=marth
P2: Python ZeroBot (zero), character=zamus

frame: 1 | me: Marth | opp: Zero Suit Samus
ZeroBot held controls: 0


In [4]:
# Sequential reset on the same environment: human versus native level-9 CPU.
obs, info = env.reset(
    players={
        1: Human(Character.Marth),
        2: CPU(Character.Samus, level=9)},
    stage=Stage.bf,
)
print(env.describe_matchup())
env.run(frames=20 * 30)
print("Completed 600 streamed frames without a bridge timeout.")

stage: battlefield
P1: human, character=marth
P2: in-game CPU level 9, character=samus
run(): 600 frames, 0 dropped
Completed 600 streamed frames without a bridge timeout.


In [5]:
# Testing contorl overlay for p1

obs, info = env.reset(
    players={
        1: Human(Character.Marth),
        2: CPU(Character.Samus, level=1)},
    stage=Stage.bf,
    render_controls=1,  # watch your inputs in the AIR window
)
print(env.describe_matchup())
print("\nPlay for a few seconds, then run the next cell...")

stage: battlefield
P1: human, character=marth
P2: in-game CPU level 1, character=samus

Play for a few seconds, then run the next cell...


Replicating human replays

In [7]:
# Record the human's held controls as a ScriptedBot script.
# Run this after playing in the cell above.
obs, info = env.reset(
    players={
        1: Human(Character.Marth),
        2: ZeroBot(Character.ZeroSuitSamus)},
    stage=Stage.bf,
)
script = env.record_human(1, frames=300)  # 10 seconds
print(f"Recorded {len(script)} entries, {sum(f for _, f in script)} frames")

# Save for later reuse
from ssf2_rl.policy.bots.scripted import save_recording, load_recording
save_recording(script, "../python/recordings/my_pattern.json")
# print("Saved to recordings/my_pattern.json")

Recorded 15 entries, 300 frames


In [6]:
# Replay the recording as a ScriptedBot
# The bot will perform exactly what the human played.
recorded_bot = ScriptedBot(Character.Marth, script, on_end="hold")
obs, info = env.reset(
    players={1: recorded_bot, 2: ZeroBot(Character.Samus)},
    stage=Stage.bf,
    render_controls=1,  # verify the bot's inputs match what you played
)
print(env.describe_matchup())
env.run(frames=300)

stage: battlefield
P1: Python ScriptedBot (scripted), character=marth
P2: Python ZeroBot (zero), character=samus
run(): 300 frames, 0 dropped


{}

In [ ]:
# --- Episode collection (game-side buffering) --------------------------------
# Collect a full episode with game-side buffering, then replay it for debugging.
# The game buffers frames internally and bulk-transfers at the end.

from ssf2_rl.data.episode import Episode

# Collect an episode with a scripted bot
episode = env.collect_episode(
    players={
        1: ScriptedBot(Character.Marth, [(RIGHT, 100), (LEFT, 100), (NOOP, 100)]),
        2: ZeroBot(Character.Samus),
    },
    stage=Stage.bf,
    frames=300,
)
print(f"Collected {len(episode)} frames")

# Save for later
episode.save("recordings/demo_episode.json")
print("Saved to recordings/demo_episode.json")

# Extract BC dataset
obs, actions = episode.to_bc_dataset(player_id=1)
print(f"BC dataset: {obs.shape[0]} samples, {obs.shape[1]} features")

stage: finaldestination
P1: Python ScriptedBot (scripted), character=marth
P2: Python ZeroBot (zero), character=samus
run(): 450 frames, 5696 dropped


In [ ]:
# --- Replay the episode for visual debugging ---------------------------------
# Load a saved episode and step through it with the overlay showing controls.

loaded = Episode.load("recordings/demo_episode.json")
print(f"Loaded {len(loaded)} frames")

# Replay in lockstep mode (slow, but you can watch each frame)
lockstep_env = SSF2Env(lockstep=True, lockstep_mode="synchronous")
try:
    loaded.replay(lockstep_env, slot=1, render_controls=1)
finally:
    lockstep_env.close()

In [14]:
# --- Fast exact lockstep -----------------------------------------------------
# Close the real-time client before opening the one supported active client.
env.close()
lockstep_env = SSF2Env(
    lockstep=True,
    lockstep_mode="synchronous",
    state_transport="json",  # compatibility default; benchmark separately
)
obs, info = lockstep_env.reset(
    players={
        1: Agent(Character.Marth),
        2: ZeroBot(Character.Samus),
    },
    stage=Stage.bf,
)
assert info["lockstep"] and info["paused"]
previous = info["frame"]

for _ in range(301):
    obs, reward, terminated, truncated, info = lockstep_env.step(0)
    assert info["paused"] and info["frame"] == previous + 1
    previous = info["frame"]

print(f"Paused at frame {info['frame']} after 301 exact policy steps.")
lockstep_env.close()

Paused at frame 302 after 301 exact policy steps.


In [5]:
# --- Human vs FollowBot (observation sanity check) ---------------------------
# Reconnect the reusable real-time environment after closing lockstep_env.
obs, info = env.reset(
    players={
        1: Human(Character.Marth),
        2: FollowBot(Character.Samus, deadzone=30.0)
    },
    render_controls=1

)
print(env.describe_matchup())
traj = env.run(frames=600, record=True)

# FollowBot's final mask remains held after run() returns instead of reverting
# to native CPU behavior.

stage: finaldestination
P1: human, character=marth
P2: Python FollowBot (follow), character=samus
run(): 600 frames, 5696 dropped
